### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response

APIConnectionError: Connection error.

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to use the get_weather function.', 'tool_calls': [{'id': 'fc_7189ce34-0d27-4175-8562-d87d14d256ad', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 127, 'total_tokens': 160, 'completion_time': 0.036086327, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.008931908, 'prompt_tokens_details': None, 'queue_time': 0.339950909, 'total_time': 0.045018235}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d6de37e6be', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a08f57-d592-7610-b2de-7e2f39a7b1a7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7189ce34-0d27-4175-8562-d87d14d256ad', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 127, 'output_to

### Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It’s sunny in Boston!


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_7e973880-1f9b-4371-ace8-985553ed8121', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 126, 'total_tokens': 158, 'completion_time': 0.034229201, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007119097, 'prompt_tokens_details': None, 'queue_time': 0.347326301, 'total_time': 0.041348298}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08f57-edf3-72c3-8d5d-8be1b37b24ea-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7e973880-1f9b-4371-ace8-985553ed8121', 'type': 'tool_cal